# Dataset Preparation project-wise 80-20 percent

In [ ]:
#Split into train and test dataset
import pandas as pd
import numpy as np
from sklearn.utils import shuffle

df = pd.read_csv('../data/comment.csv')
#Filter out comments containing  non ASCII characters
df = df[df['comment'].str.match('^[\x00-\x7F]*$', na=False)]
df = df[df['repository'] != 69]
df = shuffle(df)
# df = df.sample(frac=1, random_state=43).reset_index(drop=True)
duplicated_df = df.copy(deep=True)
unique_df = df.copy(deep=True)
unique_df['comment_lower'] = unique_df['comment'].str.strip().str.lower()
unique_df = unique_df.drop_duplicates(subset="comment_lower", keep="first").reset_index(drop=True)
unique_df.drop(columns=['comment_lower'], inplace=True)

project_ids = np.random.permutation(duplicated_df["repository"].dropna().astype(int).project_size())
for (dataset_prefix, df) in [['duplicate', duplicated_df], ['unique', unique_df]]:
    total_nx = len(df[df['satd'] == 'yes'])
    total_ny = len(df[df['satd'] == 'no'])
    test_x = 0
    test_y = 0

    df_train = pd.DataFrame(columns=df.columns)
    df_test = pd.DataFrame(columns=df.columns)
    train_project_ids = set()
    test_project_ids = set()
    for pid in project_ids:
        px_df = df[(df['repository'] == pid) & (df['satd'] == 'yes')]
        py_df = df[(df['repository'] == pid) & (df['satd'] == 'no')]
        if len(df_test[df_test['satd'] == 'yes']) < 0.20 * total_nx:
            df_test = pd.concat([df_test, px_df])
            test_x += len(px_df)
            test_project_ids.add(pid)
        else:
            df_train = pd.concat([df_train, px_df])
            train_project_ids.add(pid)

        if len(df_test[df_test['satd'] == 'no']) < 0.20 * total_ny:
            df_test = pd.concat([df_test, py_df])
            test_y += len(py_df)
            test_project_ids.add(pid)
        else:
            df_train = pd.concat([df_train, py_df])
            train_project_ids.add(pid)
    df_train = shuffle(df_train)
    df_train.drop(columns=['type'], inplace=True)
    df_train.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)
    df_train.to_csv(f'../data/{dataset_prefix}_detect_train.csv', index=False)

    df_test = shuffle(df_test)
    df_test.drop(columns=['type'], inplace=True)
    df_test.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)
    df_test.to_csv(f'../data/{dataset_prefix}_detect_test.csv', index=False)

    print(f'-------------- {dataset_prefix} Detect-------------')
    print(f'common project ids {set(map(int, train_project_ids)) & set(map(int, test_project_ids))}')
    print(f'total {len(df[df["satd"] == "yes"])}/{len(df)} of {df["repository"].nunique()} projects')
    print(
        f'train {len(df_train[df_train["label"] == "yes"])}/{len(df_train)} of {df_train["repository"].nunique()} projects')
    print(
        f'test {len(df_test[df_test["label"] == "yes"])}/{len(df_test)} of {df_test["repository"].nunique()} projects')

    #
    #
    df = df[df['satd'] == 'yes']
    df = shuffle(df)
    # df = df.sample(frac=1, random_state=43).reset_index(drop=True)
    print(f'------------- {dataset_prefix} Classification {len(df)} - {len(df["type"].project_size())}-------------')
    for t in df['type'].project_size():
        count = len(df[df['type'] == t])
        print(f"{t}: {count}/{len(df)}")
    #
    #
    split_index = int(len(df) * 0.0)
    df1 = df.iloc[:split_index]
    df2 = df.iloc[split_index:]

    df1 = df1.drop(columns=['satd'])
    df1.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)

    df2 = df2.drop(columns=['satd'])
    df2.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)

    df1.to_csv(f'../data/{dataset_prefix}_classify_train.csv', index=False)
    df2.to_csv(f'../data/{dataset_prefix}_classify_test.csv', index=False)
    df = df.drop(columns=['satd', 'code_before', 'code_after', 'code_method'])
    df.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)
    df.to_csv(f'../data/{dataset_prefix}_satd_comment.csv', index=False)

In [ ]:
import pandas as pd

DATASET_NAME = 'duplicate'

classify_train_df = pd.read_csv(f'../data/{DATASET_NAME}_classify_train.csv')
classify_test_df = pd.read_csv(f'../data/{DATASET_NAME}_classify_test.csv')

train_ids = set(classify_train_df['text'])
filtered_test_df = classify_test_df[~classify_test_df['text'].isin(train_ids)]

filtered_test_df.to_csv(f'../data/{DATASET_NAME}_classify_test.csv', index=False)


Add hash column

In [ ]:
import os
import pandas as pd
from util import *

for base_dir in ['../data', '../cache/output/merged', '../cache/output/mismatched', '../cache/output/snapshot']:
    files = list(
        filter(lambda file: file.endswith('.csv') and not file.endswith('repository.csv'), os.listdir(base_dir)))
    for file in map(lambda file: os.path.join(base_dir, file), files):
        df = pd.read_csv(file)
        print(file)
        if 'hash' not in df.columns:
            if 'text' in df.columns:
                target_column = 'text'
            elif 'comment' in df.columns:
                target_column = 'comment'
            else:
                target_column = None
            df.insert(df.columns.get_loc(target_column) + 1, 'hash', df[target_column].map(lambda x: sha1(x)))
            df.to_csv(file, index=False)


In [ ]:
#Logistic Regression
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import shuffle

under_sampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
lr_df_all = pd.read_csv('../data/duplicate_detect_train.csv')
lr_df_all = shuffle(lr_df_all)
indices, _ = under_sampler.fit_resample(lr_df_all.index.values.reshape(-1, 1), lr_df_all['label'])
lr_df_resampled = lr_df_all.loc[indices.flatten()]
lr_df_resampled.reset_index(drop=True)
lr_df_resampled = shuffle(lr_df_resampled)
lr_df_resampled.to_csv('../data/duplicate_balance_detect_train.csv', index=False)
# lr_dataset = Dataset.from_pandas(lr_df_resampled)

# Comment Count

In [ ]:
import pandas as pd

final_comments = len(pd.read_csv('../data/duplicate_detect_train.csv')) + len(
    pd.read_csv('../data/duplicate_detect_test.csv'))


df = pd.read_csv('../data/comment.csv')
df_ascii = df[df['comment'].str.match('^[\x00-\x7F]*$', na=False)]
df_ascii_without_open_ai = df_ascii[df_ascii['repository'] != 69]
#Filter out comments containing  non ASCII characters
print(f"Total : {len(df)} comments")
print(f"Total ASCII : {len(df_ascii)} comments")

print(f"Total non-ASCII : {len(df) - len(df_ascii)} comments")
print(f"Open AI ASCII comments: {len(df_ascii[df_ascii['repository'] == 69])} comments")
print(f"Open AI comments: {len(df[df['repository'] == 69])} comments")
print(f"Open AI non-ASCII comments: {len(df[df['repository'] == 69]) - len(df_ascii[df_ascii['repository'] == 69])} comments")

print(f"Total non-ASCII SATD : {len(df_ascii[df_ascii['satd'] == 'yes'])} comments")
print(f"Final projects  : {df_ascii_without_open_ai['repository'].nunique()} projects")
print(f"Final  comments: {len(df_ascii_without_open_ai)} = {final_comments}")
print(f"Final SATD  comments: {len(df_ascii_without_open_ai[df_ascii_without_open_ai['satd'] == 'yes'])}")


# Traing and Testing Project

In [ ]:
import pandas as pd
datasets = [['duplicate', 'train', pd.read_csv('../data/duplicate_detect_train.csv')],
            ['duplicate', 'test', pd.read_csv('../data/duplicate_detect_test.csv')],
            ['unique', 'train', pd.read_csv('../data/unique_detect_train.csv')],
             ['unique', 'test', pd.read_csv('../data/unique_detect_test.csv')]]
for index, [distribution, dataset_name, df] in enumerate(datasets):
    df_size = len(df)
    no_size = len(df[df['label'] == 'no'])
    yes_size = len(df[df['label'] == 'yes'])
    project_size = len(df['repository'].unique())

    other_df = datasets[index + 1 if index % 2 == 0 else index - 1][-1]
    other_df_size = len(other_df)
    other_no_size = len(other_df[other_df['label'] == 'no'])
    other_yes_size = len(other_df[other_df['label'] == 'yes'])

    overall_project_size = len(set(df['repository'].unique()).union(set(other_df['repository'].unique())))
    print(f"{distribution} & {dataset_name} & {df_size:,} ({df_size / (df_size + other_df_size) * 100:.1f}) & {no_size:,} ({no_size / (no_size + other_no_size) * 100: .1f}) & {yes_size:,} ({yes_size / (yes_size + other_yes_size) * 100:.1f}) & {yes_size/df_size*100:.1f} & {project_size} ({project_size / overall_project_size * 100:.1f}) \\\\")



In [ ]:
    import pandas as pd
    df = pd.read_csv('../data/duplicate_satd_comment.csv')
    print(f'------------- Classification {len(df)}-------------')
    for t in sorted(df['label'].unique()):
        count = len(df[df['label'] == t])
        print(f"{t}: {count}/{len(df)}")

In [ ]:
    import pandas as pd
    original_df = pd.read_csv('../data/duplicate_satd_comment_bk.csv')
    df = pd.read_csv('../data/duplicate_satd_comment.csv')
    original_df["new_label"] = df["label"]
    diff_df = original_df[original_df["new_label"] != original_df["label"]]
    diff_df.to_csv("../data/duplicate_satd_comment_diff.csv", index=False)

In [ ]:
import pandas as pd
df = pd.read_csv('../data/duplicate_satd_comment.csv')
df['label'] = df['label'].map(lambda x : 'composite' if x == 'multi' else x)
df.to_csv('../data/duplicate_satd_comment.csv', index=False)